In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
merged_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/all_metrics_1and2percent.csv')

In [3]:
cols = [
    'setting', 'method', 'bio_sim_300',
    'category', 'top_recall_300',
    'bedroc_300', 'auroc'
]

cdf = merged_df[cols].copy()

cdf = (
    cdf.groupby(['setting', 'method', 'category'], as_index=False)
      .mean(numeric_only=True)
)

In [4]:
map_name = {'2019_mf_add_pred':'MF', 
       '2019_nn_all_pred':'DNN',
       '2019_rf_renamed_pred':'RF',
       'df_gnn_occsvm_pred':'SVM',
       'gcn':'GCN', 
       'graphsage':'GraphSage'
}
cdf = cdf[cdf['setting'].isin(list(map_name.keys()))]
cdf['model'] = cdf['setting'].map(map_name)

In [5]:
cdf = cdf[cdf['method'].isin(['literature_emb', 'ppi_emb_df', 'ppi_emb_dw', 'ppi_emb_n2v',
       'seq_emb_esm', 'seq_emb_port'])]

In [6]:
cdf['feature'] = cdf['method']

# feature discovery
give a subdf ['feature', 'category', 'bio_sim_300', 'top_recall_300','bedroc_300', 'auroc']
1. in which category, 'literature_emb' better than features start with 'ppi' in at least 2 metrics
2. in which category, features start with 'seq' better than features start with 'ppi' in at least 1 metrics
3. in which category, features start with 'seq' better than 'literature_emb' in at least 1 metrics
4. in which category, features start with 'ppi' always better than others in all metrics
5. in which category, features start with 'ppi'  better than others in 3 metrics except 1, and what the expection is?

In [25]:
import pandas as pd
from collections import defaultdict

RECALL_BEDROC = ["top_recall_300", "bedroc_300"]
BIO_SIM       = ["bio_sim_300"]
AUROC         = ["auroc"]
ALL_METRICS   = RECALL_BEDROC + BIO_SIM + AUROC

MARGIN        = 0.1   # normalized difference threshold for "better"


def minmax_normalize(df):
    """
    MinMax-normalize ALL_METRICS globally across the full dataframe.
    Operates on a copy; original df is unchanged.
    Range is computed across all rows (all models, categories, features).
    """
    df = df.copy()
    for col in ALL_METRICS:
        mn, mx = df[col].min(), df[col].max()
        if mx > mn:
            df[col] = (df[col] - mn) / (mx - mn)
        else:
            df[col] = 0.0   # constant column — no information
    return df


def agg(df, prefix=None, exact=None):
    mask = (df["feature"].str.startswith(prefix) if prefix
            else df["feature"] == exact)
    return (df[mask]
            .groupby("category")[ALL_METRICS]
            .max()
            .rename(columns=lambda c: f"{prefix or exact}__{c}"))


def better_in(merged, a, b, cols):
    """A is 'better' if normalized(A) - normalized(B) > MARGIN for ALL cols."""
    combined = (merged[f"{a}__{cols[0]}"] - merged[f"{b}__{cols[0]}"]) > MARGIN
    for c in cols[1:]:
        combined = combined & (
            (merged[f"{a}__{c}"] - merged[f"{b}__{c}"]) > MARGIN)
    return set(merged[combined].index)


def _compare_cases(merged, a, b):
    win_rb  = better_in(merged, a, b, RECALL_BEDROC)
    win_bs  = better_in(merged, a, b, BIO_SIM)
    lose_rb = better_in(merged, b, a, RECALL_BEDROC)
    auroc_cols = [f"{a}__auroc", f"{b}__auroc"]
    return {
        "a": (sorted(win_rb & win_bs),  None),
        "b": (sorted(win_rb - win_bs),  merged[auroc_cols]),
        "c": (sorted(lose_rb & win_bs), merged[auroc_cols]),
    }


def _ppi_dom_cases(df):
    ppi_max = (df[df["feature"].str.startswith("ppi")]
               .groupby("category")[ALL_METRICS].max())
    others  = (df[~df["feature"].str.startswith("ppi")]
               .groupby("category")[ALL_METRICS].max())
    j = ppi_max.join(others, lsuffix="_ppi", rsuffix="_other", how="inner")

    def dom(cols):
        combined = (j[f"{cols[0]}_ppi"] - j[f"{cols[0]}_other"]) > MARGIN
        for c in cols[1:]:
            combined = combined & (
                (j[f"{c}_ppi"] - j[f"{c}_other"]) > MARGIN)
        return set(j[combined].index)

    win_rb = dom(RECALL_BEDROC)
    win_bs = dom(BIO_SIM)
    return {
        "4a": sorted(win_rb & win_bs),
        "4b": sorted(win_rb - win_bs),
        "4c": sorted(win_bs - win_rb),
    }


def run_all_models(cdf, margin=MARGIN):
    """
    Parameters
    ----------
    cdf    : raw dataframe with column 'model'
    margin : override the global MARGIN threshold at call time if needed

    Returns result_df with columns:
        category | question | models | auroc_detail | model_count
    """
    global MARGIN
    MARGIN = margin

    norm_cdf = minmax_normalize(cdf)   # normalize once globally

    collector = defaultdict(lambda: {"models": [], "auroc_detail": {}})

    for model in norm_cdf["model"].unique():
        subdf = norm_cdf[norm_cdf["model"] == model].copy()

        pairs = [
            ("Q1", "literature_emb", None,  None,            "ppi"),
            ("Q2", None,             "seq", None,            "ppi"),
            ("Q3", None,             "seq", "literature_emb", None),
        ]
        for qid, a_exact, a_prefix, b_exact, b_prefix in pairs:
            a = a_exact or a_prefix
            b = b_exact or b_prefix
            try:
                merged = agg(subdf, prefix=a_prefix, exact=a_exact).join(
                         agg(subdf, prefix=b_prefix, exact=b_exact), how="inner")
            except Exception:
                continue
            for sub, (cats, auroc_df) in _compare_cases(merged, a, b).items():
                key = f"{qid}{sub}"
                for cat in cats:
                    rec = collector[(cat, key)]
                    rec["models"].append(model)
                    if auroc_df is not None and cat in auroc_df.index:
                        rec["auroc_detail"][model] = (
                            auroc_df.loc[cat].to_dict())

        for sub, cats in _ppi_dom_cases(subdf).items():
            for cat in cats:
                collector[(cat, f"Q{sub}")]["models"].append(model)

    rows = []
    for (cat, question), rec in sorted(collector.items()):
        rows.append({
            "category":     cat,
            "question":     question,
            "models":       rec["models"],
            "model_count":  len(rec["models"]),
            "auroc_detail": rec["auroc_detail"] or None,
        })

    return pd.DataFrame(rows, columns=["category", "question",
                                       "models", "model_count",
                                       "auroc_detail"])


result_df = run_all_models(cdf)

In [28]:
result_df["model_count"] = result_df["models"].str.len()

# majority rule
result_df[result_df["model_count"] >= 2]

,category,question,models,model_count,auroc_detail
1,Diseases of the blood and certain disorders(3),Q3b,"[MF, RF, SVM, GCN, GraphSage]",5,"{'MF': {'seq__auroc': 0.7011597111647999, 'lit..."
2,Diseases of the blood and certain disorders(3),Q4b,"[DNN, SVM, GraphSage]",3,None
5,Diseases of the digestive system(2),Q3b,"[MF, SVM, GraphSage]",3,"{'MF': {'seq__auroc': 0.8026978108415367, 'lit..."
6,Diseases of the digestive system(2),Q4b,"[DNN, RF]",2,None
7,Diseases of the musculoskeletal system and con...,Q4b,"[MF, DNN, RF, GraphSage]",4,None
10,Diseases of the respiratory system(3),Q4b,"[DNN, RF, SVM]",3,None
14,"Endocrine, nutritional and metabolic diseases(2)",Q4c,"[DNN, SVM, GraphSage]",3,None
15,Mental and behavioural disorders(5),Q4c,"[RF, SVM]",2,None
16,Neoplasms(8),Q1c,"[RF, SVM]",2,{'RF': {'literature_emb__auroc': 0.12094529253...
18,Neoplasms(8),Q4b,"[RF, GraphSage]",2,None


Q1a: lit_emb > ppi* recall+bedroc better AND bio_sim+auroc better
Q1b: lit_emb > ppi* recall+bedroc better, NOT bio_sim — report auroc
Q1c: lit_emb > ppi* recall+bedroc worse — bio_sim better; report auroc

Q2a: seq* > ppi* recall+bedroc better AND bio_sim+auroc better
Q2b: seq* > ppi* recall+bedroc better, NOT bio_sim — report auroc
Q2c: seq* > ppi* recall+bedroc worse — bio_sim better; report auroc

Q3a: seq* > lit_emb recall+bedroc better AND bio_sim+auroc better
Q3b: seq* > lit_emb recall+bedroc better, NOT bio_sim — report auroc
Q3c: seq* > lit_emb recall+bedroc worse — bio_sim better; report auroc

Q4a: categories where ppi* beats ALL others in every metric
Q4b: ppi* beats ALL others in recall+bedroc, NOT bio_sim
Q4c: ppi* beats ALL others in bio_sim, NOT recall+bedroc

question: 
how big different is different? e.g. in category, model A > B with 0.001 in auroc? not necessarily, so i sacaled all metrics to [0,1], if differences > 0.1 after scale, it is a differentce
how to conclude? if only some conclusion only appears in one model, is it still worth?

# model discovery
for each feature:
1. in which category, which model is better than others in all
2. in which category, which model is better than others in at least 3 metrics except 1, and what the expection is?

In [7]:
import pandas as pd

# assumes cdf contains:
# ['model', 'feature', 'category', 'bio_sim_300', 'top_recall_300', 'bedroc_300', 'auroc']

metrics = ['bio_sim_300', 'top_recall_300', 'bedroc_300', 'auroc']


# ---------- helpers ----------
def pairwise_model_wins(lrow, rrow, metrics):
    return [m for m in metrics if lrow[m] > rrow[m]]


def model_all_metrics_better(subdf, metrics):
    """
    For one feature only.
    Return:
      {category: [detailed answers]}
    where a model qualifies if it is better than every other model
    in all 4 metrics.
    """
    results = {}

    for cat, df_cat in subdf.groupby('category'):
        if df_cat['model'].nunique() < 2:
            continue

        qualified = []

        for _, mrow in df_cat.iterrows():
            flags = []
            details = []

            others = df_cat[df_cat['model'] != mrow['model']]

            for _, orow in others.iterrows():
                wins = pairwise_model_wins(mrow, orow, metrics)
                ok = len(wins) == len(metrics)  # all 4
                flags.append(ok)

                details.append({
                    'winner_model': mrow['model'],
                    'other_model': orow['model'],
                    'wins': wins
                })

            if len(flags) > 0 and all(flags):
                qualified.extend(details)

        if qualified:
            results[cat] = qualified

    return results


def model_3_of_4_with_exception(subdf, metrics):
    """
    For one feature only.
    Return:
      {category: [detailed answers]}
    where a model qualifies if it is better than every other model
    in exactly 3 out of 4 metrics, and report the exception metric.
    """
    results = {}

    for cat, df_cat in subdf.groupby('category'):
        if df_cat['model'].nunique() < 2:
            continue

        qualified = []

        for _, mrow in df_cat.iterrows():
            flags = []
            details = []

            others = df_cat[df_cat['model'] != mrow['model']]

            for _, orow in others.iterrows():
                wins = pairwise_model_wins(mrow, orow, metrics)
                exceptions = [m for m in metrics if m not in wins]
                ok = len(wins) == 3

                flags.append(ok)
                details.append({
                    'winner_model': mrow['model'],
                    'other_model': orow['model'],
                    'wins': wins,
                    'exception': exceptions
                })

            if len(flags) > 0 and all(flags):
                qualified.extend(details)

        if qualified:
            results[cat] = qualified

    return results


def flatten_model_discovery(nested_collector):
    """
    first layer only have: feature, question, category
    second layer is stored in 'details'
    """
    rows = []
    for feature, q_dict in nested_collector.items():
        for question, cat_dict in q_dict.items():
            for category, details in cat_dict.items():
                rows.append({
                    'feature': feature,
                    'question': question,
                    'category': category,
                    'details': details
                })
    return pd.DataFrame(rows)


# ---------- main collector ----------
model_discovery_collector = {}

for feature_i in cdf['feature'].dropna().unique():
    subdf = cdf.loc[cdf['feature'] == feature_i, [
        'model', 'feature', 'category',
        'bio_sim_300', 'top_recall_300', 'bedroc_300', 'auroc'
    ]].copy()

    q1 = model_all_metrics_better(subdf, metrics)
    q2 = model_3_of_4_with_exception(subdf, metrics)

    model_discovery_collector[feature_i] = {
        'q1': q1,
        'q2': q2,
    }


# ---------- outputs ----------
model_discovery_df = flatten_model_discovery(model_discovery_collector)
model_discovery_summary_df = model_discovery_df[['feature', 'question', 'category']].copy()

print(model_discovery_summary_df.head())


# ---------- optional: summary with winner model names ----------
def extract_winner_models(details):
    return sorted(set(d['winner_model'] for d in details))

model_discovery_with_winners = model_discovery_df.copy()
model_discovery_with_winners['winner_models'] = model_discovery_with_winners['details'].apply(extract_winner_models)

          feature question                                          category
0  literature_emb       q1  Endocrine, nutritional and metabolic diseases(2)
1  literature_emb       q1               Mental and behavioural disorders(5)
2  literature_emb       q2               Diseases of the digestive system(2)
3      ppi_emb_df       q1               Mental and behavioural disorders(5)
4      ppi_emb_df       q2  Endocrine, nutritional and metabolic diseases(2)


In [45]:
counts = (
    model_discovery_df[['feature', 'question', 'category']]
    .drop_duplicates()
    .groupby(['question', 'category'])['feature']
    .agg(lambda x: sorted(set(x)))
    .reset_index(name='features')
)
counts

,question,category,features
0,q1,"Endocrine, nutritional and metabolic diseases(2)","[literature_emb, seq_emb_esm, seq_emb_port]"
1,q1,Mental and behavioural disorders(5),"[literature_emb, ppi_emb_df]"
2,q2,Diseases of the digestive system(2),"[literature_emb, ppi_emb_dw, ppi_emb_n2v, seq_..."
3,q2,Diseases of the musculoskeletal system and con...,[seq_emb_esm]
4,q2,"Endocrine, nutritional and metabolic diseases(2)",[ppi_emb_df]


In [46]:
model_discovery_with_winners

,feature,question,category,details,winner_models
0,literature_emb,q1,"Endocrine, nutritional and metabolic diseases(2)","[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
1,literature_emb,q1,Mental and behavioural disorders(5),"[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
2,literature_emb,q2,Diseases of the digestive system(2),"[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
3,ppi_emb_df,q1,Mental and behavioural disorders(5),"[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
4,ppi_emb_df,q2,"Endocrine, nutritional and metabolic diseases(2)","[{'winner_model': 'MF', 'other_model': 'DNN', ...",[MF]
5,ppi_emb_dw,q2,Diseases of the digestive system(2),"[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
6,ppi_emb_n2v,q2,Diseases of the digestive system(2),"[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
7,seq_emb_esm,q1,"Endocrine, nutritional and metabolic diseases(2)","[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
8,seq_emb_esm,q2,Diseases of the musculoskeletal system and con...,"[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
9,seq_emb_port,q1,"Endocrine, nutritional and metabolic diseases(2)","[{'winner_model': 'GCN', 'other_model': 'MF', ...",[GCN]
